In [ ]:
from langgraph.graph import START, StateGraph, END
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from src.cv_model.resnet_model import build_model
import numpy as np
from pathlib import Path
import os
import cv2

In [2]:
import sys
from pathlib import Path

# go one folder up from research/
project_root = Path("../").resolve()

sys.path.append(str(project_root))

In [4]:
load_dotenv()

True

In [5]:
# model
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

In [6]:
# helper function for model tool
IMG_SIZE = (224, 224)

def crop_center_region_np(image):
    h, w = image.shape[:2]

    top_crop = int(0.05 * h)
    bottom_crop = int(0.05 * h)
    left_crop = int(0.08 * w)
    right_crop = int(0.08 * w)

    cropped = image[
        top_crop:h - bottom_crop,
        left_crop:w - right_crop,
        :
    ]

    cropped = cv2.resize(cropped, IMG_SIZE)
    return cropped

def apply_clahe_np(image):
    image = np.uint8(np.clip(image, 0, 255))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(gray)

    clahe_img = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2RGB)
    return clahe_img.astype(np.float32)

def preprocess_xray(image_path):
    try:
        image_path = str(image_path)
        # read image
        image = cv2.imread(image_path)

        # BGR > RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # resize
        image = cv2.resize(image, IMG_SIZE)
        # crop
        image = crop_center_region_np(image)
        # CLAHE
        image = apply_clahe_np(image)
        # ResNet preprocessing
        image = preprocess_input(image)
        # add batch dimension
        image = np.expand_dims(image, axis=0)
    except Exception as e:
        print("Exception Occured: ",e)
    return image


In [7]:
def predict_xray(model,image_path):
    class_names = ['Normal', 'Tuberculosis']
    x = preprocess_xray(image_path=image_path)
    prob = model.predict(x)[0][0]
    
    if prob >= 0.35:
        pred_class = 1
        pred_label = class_names[1]
    else:
        pred_class = 0
        pred_label = class_names[0]
    
    confidence = prob if pred_class == 1 else 1 - prob
        
    return {
        "probability_tuberculosis": float(prob),
        "prediction" : pred_label,
        "confidence": float(confidence)
    }

In [9]:
# testing the model with samples
image_path = Path("../test_images/img_normal1.jpg")
image_model_weights_path = Path("../CV_model") / "tb_weights_fixed.weights.h5"

model = build_model()

model.load_weights(image_model_weights_path)

In [10]:
predict_xray(model,image_path=image_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


{'probability_tuberculosis': 0.0012055817060172558,
 'prediction': 'Normal',
 'confidence': 0.998794436454773}

## Grad-cam tool


In [11]:
def make_gradcam_heatmap(img_array, model, pred_index=None):

    # correct layers
    base_model = model.layers[1]

    gap = model.layers[2]
    dense1 = model.layers[3]
    dropout1 = model.layers[4]
    dense2 = model.layers[5]
    dropout2 = model.layers[6]
    out_layer = model.layers[7]

    with tf.GradientTape() as tape:

        # feature maps from ResNet50
        conv_outputs = base_model(img_array)

        tape.watch(conv_outputs)

        # classifier head
        x = gap(conv_outputs)

        x = dense1(x)

        x = dropout1(
            x,
            training=False
        )

        x = dense2(x)

        x = dropout2(
            x,
            training=False
        )

        preds = out_layer(x)

        # binary classifier
        class_channel = preds[:, 0]

    # gradients
    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    # average gradients
    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    # weighted activation maps
    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    # ReLU
    heatmap = tf.maximum(
        heatmap,
        0
    )

    # normalize
    heatmap /= (
        tf.reduce_max(heatmap)
        + tf.keras.backend.epsilon()
    )

    return heatmap.numpy()


def detect_highlighted_region(heatmap):
    h, w = heatmap.shape

    y, x = np.unravel_index(np.argmax(heatmap), heatmap.shape)

    if y < h / 3:
        vertical = "upper"
    elif y < 2 * h / 3:
        vertical = "middle"
    else:
        vertical = "lower"

    if x < w / 3:
        horizontal = "left"
    elif x > 2 * w / 3:
        horizontal = "right"
    else:
        horizontal = "central"

    return f"{vertical} {horizontal} lung region"

def preprocess_image(image_path):
    try:
        image_path = str(image_path)

        # read image
        image = cv2.imread(image_path)

        # BGR -> RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # resize
        image = cv2.resize(image, IMG_SIZE)

        # save original for Grad-CAM overlay
        original_image = image.copy()

        # crop
        image = crop_center_region_np(image)

        # CLAHE
        image = apply_clahe_np(image)

        # ResNet preprocessing
        image = preprocess_input(image)

        # add batch dimension
        image = np.expand_dims(image, axis=0)

        return original_image, image

    except Exception as e:
        print("Exception Occured:", e)
        return None, None

In [12]:
def gradcam_tool(image_path,cv_model):
    # create output directory
    os.makedirs("grad_cam_outputs", exist_ok=True)

    # preprocess image
    original_img, img_array = preprocess_image(image_path)

    # generate heatmap
    heatmap = make_gradcam_heatmap(img_array, model)

    # resize heatmap
    heatmap_resized = cv2.resize(
        heatmap,
        (original_img.shape[1], original_img.shape[0])
    )

    # convert heatmap to color
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(
        heatmap_uint8,
        cv2.COLORMAP_JET
    )

    # overlay
    overlay = cv2.addWeighted(
        original_img.astype(np.uint8),
        0.6,
        heatmap_color,
        0.4,
        0
    )

    # save image
    output_path = (
        f"grad_cam_outputs/"
        f"{Path(image_path).stem}_gradcam.png"
    )

    cv2.imwrite(output_path, overlay)

    # highlighted region
    highlighted_region = detect_highlighted_region(heatmap)

    return {
        "grad_cam": output_path,
        "highlighted_region": highlighted_region
    }

In [14]:
image_paths = [Path('../test_images/n2.jpg'),Path('../test_images/tb1.gif'),Path('../test_images/tb2.jpg')]

for path in image_paths:
    output = gradcam_tool(path,model)
    print(output)

{'grad_cam': 'grad_cam_outputs/n2_gradcam.png', 'highlighted_region': 'middle left lung region'}
{'grad_cam': 'grad_cam_outputs/tb1_gradcam.png', 'highlighted_region': 'middle left lung region'}
{'grad_cam': 'grad_cam_outputs/tb2_gradcam.png', 'highlighted_region': 'upper left lung region'}


## Symptom Collection Tool

In [15]:
TB_SYMPTOMS = [
    "cough",
    "fever",
    "weight loss",
    "night sweats",
    "chest pain",
    "fatigue",
    "breathing difficulty",
    "blood in sputum"
]


def symptom_collection_tool(user_input):

    user_input = user_input.lower()

    detected_symptoms = []

    for symptom in TB_SYMPTOMS:

        if symptom in user_input:
            detected_symptoms.append(symptom)

    return {
        "detected_symptoms": detected_symptoms,
        "symptom_count": len(detected_symptoms)
    }
    

## RAG tool


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings

In [17]:
def data_loadeer_and_splitter(data_path):
    
    loader = PyPDFLoader(data_path)
    docs = loader.load()
    
    for doc in docs:
        doc.metadata = {
            'source':'Handbook of Tuberculosis',
            'Editors': 'Jacques H. Grosset and Richard E. Chaisson',
            'page': doc.metadata['page'],
            'total_pages': doc.metadata['total_pages']
            }
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100
        )
    
    chunks = splitter.split_documents(docs)
    
    return chunks

In [ ]:
from pathlib import Path

data_path = Path("../data/109.pdf")

loader = PyPDFLoader(data_path)
docs = loader.load()

for doc in docs:
    doc.metadata = {
        'source':'Handbook of Tuberculosis',
        'Editors': 'Jacques H. Grosset and Richard E. Chaisson',
        'page': doc.metadata['page'],
        'total_pages': doc.metadata['total_pages']
    }
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
    )
chunks = splitter.split_documents(docs)

In [24]:
embedding_model = HuggingFaceEmbeddings(
     model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2574.96it/s]


In [26]:
def data_ingestion(indexname,chunks,embedding_model):
    pc = Pinecone()
    if not pc.has_index(indexname):
        pc.create_index(
            name=indexname,
            dimension=384,
            metric='cosine',
            spec=ServerlessSpec(
                cloud='aws',
                region='us-east-1'
            )
        )
    docsearch = PineconeVectorStore.from_documents(
        documents=chunks,
        index_name=indexname,
        embedding=embedding_model,
    )
    
    print("Data Succesfully uploded to pinecone")
    

In [25]:
def create_vectorstore(indexname,embedding_model):
    vectorstore = PineconeVectorStore.from_existing_index(
        index_name=indexname,
        embedding=embedding_model
    )
    return vectorstore

In [27]:
def get_retriever(vectordb,chunks):
    
    # pineconer search kwargs
    search_kwargs = {
        "k":5,
        'fetch_k':20,
        "lambda_mult":0.5
    }
    
    pinecone_retriever = vectordb.as_retriever(
        search_type="mmr",
        search_kwargs=search_kwargs
    )
    
    # BM25 retriever
    bm25_retriever = BM25Retriever.from_documents(chunks)
    
    bm25_retriever.k = 3
    
    
    # hybrid retiever
    hybrid_retiever = EnsembleRetriever(
        retrievers=[pinecone_retriever,bm25_retriever],
        weights=[0.8,0.2]
    )
    
    return hybrid_retiever

In [29]:
index_name = "tb-chunks"

data_ingestion(indexname=index_name,chunks=chunks,embedding_model=embedding_model)

Data Succesfully uploded to pinecone


In [30]:
vector_db = create_vectorstore(indexname=index_name,embedding_model=embedding_model)
retriever = get_retriever(vectordb=vector_db,chunks=chunks)

In [33]:
retriever.invoke("what is Tb its preventations?")

[Document(metadata={'Editors': 'Jacques H. Grosset and Richard E. Chaisson', 'page': 26.0, 'source': 'Handbook of Tuberculosis', 'total_pages': 236.0}, page_content='ics of human susceptibility to TB is a topic under considerable \nstudy.\n1.4  Epidemiology\nThe natural history of tuberculosis begins with infection of a \nhost, establishment of latent or subclinical infection, and pro-\ngression to active disease, which occurs in only a small pro-\nportion of those infected [21]. The organism is transmitted by \nthe airborne route when individuals with active respiratory \ntract infections expel bacilli into the environment. Humans \nare both the obligate host and vector of M. tuberculosis, and \nno other reservoirs of infection exist. M. bovis causes disease'),
 Document(metadata={'Editors': 'Jacques H. Grosset and Richard E. Chaisson', 'page': 164.0, 'source': 'Handbook of Tuberculosis', 'total_pages': 236.0}, page_content='intervention. Occult hematogenous spread is thought to \nocc